# Multiple fuels: octane primary + hydrogen reheat

Two **very different fuels injected at different positions** and burned:
liquid-like **n-octane** (`C8H18`) in the primary zone, then **hydrogen**
(`H2`) injected into the hot products as a **reheat** stage.

This shows the chemistry plumbing is fuel-agnostic:

* you name whatever species you want from `thermo.inp`; the network transports
  one conserved **mixture fraction** per distinct injected composition (here
  air, octane, hydrogen), worked out from the element compositions at build time;
* a source specified by species needs no translation table -- its stream is
  reconstructed exactly by a forward blend, so octane and hydrogen coexist with
  no element bookkeeping;
* injecting H2 into the **burnt** (equilibrium) stream re-equilibrates and
  releases more heat -- a reheat with no extra flame element needed.

> **Mixture-fraction transport.** Each feed stream is its own transported scalar,
> so the unburnt reconstruction is unambiguous no matter how many fuels are
> co-mixed -- even two carbon-bearing fuels injected into one unburnt stream
> (which an elemental basis cannot resolve) is fine here.

In [ ]:
import os, sys
# add the repo root (the dir containing the `nefes` package) to sys.path
_root = os.getcwd()
while not os.path.isdir(os.path.join(_root, "nefes")) and _root != os.path.dirname(_root):
    _root = os.path.dirname(_root)
sys.path.insert(0, _root)

import numpy as np
import matplotlib.pyplot as plt

from thermolib import ThermoInp, Thermo
from nefes.chem.composition import resolve_composition, enthalpy_mass
from nefes.elements import catalog as cat
from nefes.solver import solve
from nefes.solver.control import states_table
from nefes.thermo.api import EQ_FROZEN, EQ_KERNEL
from nefes.thermo.configure import equilibrium
from nefes.assembly.derive import ES_MDOT, ES_P, ES_HT, ES_RHO, ES_U, ES_T, ES_M, ES_PT
from nefes.shell import Network

THERMO_INP = os.path.join(_root, "thermolib", "data", "thermo.inp")
RU = 8.31446261815324

In [ ]:
HEAVY = "C8H18,n-octane"
lib = ThermoInp(THERMO_INP).library(
    ["O2", "N2", HEAVY, "H2", "CO2", "H2O", "CO", "OH", "H", "O", "NO"]
)
gas = Thermo(lib)
print("elements:", lib.elements)
print("species :", [s.name for s in lib.species])

## Staged network
`air -> octane injector -> flame -> H2 reheat injector -> outlet`.  The octane
burns at the flame (frozen -> equilibrium); the H2 is injected into the burnt
equilibrium stream and re-equilibrates (reheat).

In [ ]:
AIR = {"O2": 0.21, "N2": 0.79}
OCT = {HEAVY: 1.0}
H2 = {"H2": 1.0}

def build_staged(mdot_air=1.0, mdot_oct=0.04, mdot_h2=0.008, Tair=400.0, Tfuel=300.0, p=5.0e5, A=0.06):
    """air -> octane injector -> flame -> H2 reheat injector -> outlet.

    Three feed streams (air, octane, H2) are discovered at build time; `solve(prob)`
    seeds every edge by propagating them through the network -- no hand-built guess.
    """
    Yair, _ = resolve_composition(lib, AIR)
    Yoct, _ = resolve_composition(lib, OCT)
    h_air = enthalpy_mass(lib, Yair, Tair)
    h_oct = enthalpy_mass(lib, Yoct, Tfuel)
    m1 = mdot_air + mdot_oct
    m2 = m1 + mdot_h2
    h_mix = (mdot_air * h_air + mdot_oct * h_oct) / m1  # enthalpy scale for h_ref

    cfg = equilibrium(lib)
    els = [
        cat.mass_flow_inlet(mdot_air, Tair, composition=AIR, name="air"),
        cat.mass_source(mdot_oct, Tfuel, composition=OCT, name="octane"),
        cat.equilibrium_flame(name="flame"),
        cat.mass_source(mdot_h2, Tfuel, composition=H2, name="H2-reheat"),
        cat.pressure_outlet(p, Tt_backflow=Tair, composition=AIR, name="out"),
    ]
    edges = [(0, 1, A), (1, 2, A), (2, 3, A), (3, 4, A)]
    edge_models = [EQ_FROZEN, EQ_FROZEN, EQ_KERNEL, EQ_KERNEL]
    network = Network(cfg, p_ref=p, mdot_ref=m2, h_ref=abs(h_mix), nodes=els, edges=edges,
                      edge_models=edge_models)
    prob = cat.build_problem(cfg, els, edges, mdot_ref=m2, p_ref=p, h_ref=abs(h_mix),
                             edge_models=edge_models)
    return prob, network

prob, network = build_staged()
res = solve(prob)
# the feed streams are discovered at build time, so the labels live on the problem
print("feed streams (transported mixture fractions):", list(prob.scalar_names))
print("converged:", res.converged, " iterations:", res.iterations)

## Edge states and the feed-stream mixture fractions
The transported mixture fractions `xi` pick up octane as it is injected and H2 at the reheat stage; the flame burns octane, the H2 stage reheats.

In [ ]:
est = states_table(prob, res.x)
names = ["air", "air+octane", "burnt(oct)", "burnt+H2"]
print(f"{'edge':<12}{'mdot':>9}{'T [K]':>10}{'p [bar]':>10}{'M':>8}")
for e in range(prob.n_edges):
    print(f"{names[e]:<12}{est[ES_MDOT,e]:9.4f}{est[ES_T,e]:10.1f}{est[ES_P,e]/1e5:10.3f}{est[ES_M,e]:8.3f}")
print()
# transported mixture fractions xi per edge (rows 3..) -- one per feed stream
xi = res.x[3:, :]
print("feed-stream mixture fractions xi by edge:")
print(f"{'':<12}" + "".join(f"{nm:>11}" for nm in prob.scalar_names))
for e in range(prob.n_edges):
    print(f"{names[e]:<12}" + "".join(f"{xi[i,e]:11.4f}" for i in range(prob.n_elem)))
print(f"\nH2 reheat raises T by {est[ES_T,3]-est[ES_T,2]:.0f} K")

## Temperature through the stages

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
xa = [0, 1, 2, 3]; T = [est[ES_T, e] for e in range(4)]
ax.plot(xa, T, "o-", lw=2, ms=8, color="#8e44ad")
for xi, ti, nm in zip(xa, T, names):
    ax.annotate(f"{nm}\n{ti:.0f} K", (xi, ti), textcoords="offset points", xytext=(0, 10), ha="center", fontsize=9)
ax.set_xticks(xa); ax.set_xticklabels(["air", "octane in", "flame", "H2 reheat"])
ax.set_ylabel("static temperature [K]"); ax.set_title("Octane primary + H2 reheat")
ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

## Hydrogen-reheat sweep
More H2 into the burnt gas -> more reheat (until the residual oxygen is consumed).

In [ ]:
h2s = np.linspace(0.0, 0.02, 11)
Treheat, Tprimary = [], []
for mh in h2s:
    prob_i, _ = build_staged(mdot_h2=max(mh, 1e-9))
    r = solve(prob_i)
    e = states_table(prob_i, r.x)
    Tprimary.append(e[ES_T, 2]); Treheat.append(e[ES_T, 3])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(h2s, Tprimary, "s--", label="after octane flame", color="#c0392b")
ax.plot(h2s, Treheat, "o-", label="after H2 reheat", color="#8e44ad")
ax.set_xlabel("H2 reheat mdot [kg/s]"); ax.set_ylabel("temperature [K]")
ax.set_title("Hydrogen-reheat sweep"); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## Export for FNetLibUI

The full network is available as `network` (a `Network`) and its converged mean flow as `sol` (a `Solution`).
Save either to a UI-readable YAML — `sol.to_yaml(path)` embeds the mean-flow result fields, `network.to_yaml(path)` writes the topology only — then open the file in FNetLibUI.

In [ ]:
import os, tempfile

sol = network.solve()  # the primary staged network, re-solved via the shell
_out = os.path.join(tempfile.mkdtemp(), "multiple_fuels.yaml")
sol.to_yaml(_out)  # embeds the mean-flow results; use network.to_yaml(_out) for topology only
print("saved case:", _out)